In [1]:
import os
import json
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder

In [2]:
BASE_DIR = 'PartImageNet_Seg/PartImageNet'
ANNOTATIONS_DIR = os.path.join(BASE_DIR, 'annotations')
IMAGES_DIR = os.path.join(BASE_DIR, 'images')

# --- Training Configuration ---
# You can choose which dataset to use in the main function
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 1e-4

In [32]:
# TARGET_CLASSES = [
#     'n02691156',  # airplane
#     'n02131653',  # bear
#     'n02834778',  # bicycle
#     'n01503061',  # bird
#     'n02858304',  # boat
#     'n02876657',  # bottle
#     'n02958343',  # car
#     'n02121808',  # cat
#     'n03001627',  # chair
#     'n03046257',  # clock
#     'n02084071',  # dog
#     'n02503517',  # elephant
#     'n03627232',  # keyboard
#     'n03633091',  # knife
#     'n03874599',  # oven
#     'n04468005'   # truck
# ]

NUM_PARTS = 10 # The value you found using the analysis script

# The total number of channels will be the parts + 1 for the global image
TOTAL_CHANNELS = NUM_PARTS + 1

## Simple CNN with independent channel

In [4]:
class PartImageNetCOCODataset(Dataset):
    """
    Loads data from a single COCO JSON, and correctly constructs the path to images
    that are directly inside the split folder (e.g., 'images/train/').
    """
    def __init__(self, coco_json_path, image_split_dir, num_parts, img_dims):
        self.coco_json_path = coco_json_path
        self.image_split_dir = image_split_dir # The specific split dir, e.g., '.../images/train'
        self.num_parts = num_parts
        self.img_height, self.img_width = img_dims
        self.data = self._load_coco_data()
        self.image_ids = list(self.data['images'].keys())

    def _load_coco_data(self):
        if not os.path.exists(self.coco_json_path):
            raise FileNotFoundError(f"❌ COCO JSON file not found at: {self.coco_json_path}")
        print(f"Loading annotations from: {self.coco_json_path}...")
        with open(self.coco_json_path, 'r') as f:
            raw_data = json.load(f)
        images = {img['id']: img for img in raw_data['images']}
        annotations_by_image = {img_id: [] for img_id in images}
        for ann in raw_data['annotations']:
            if ann['image_id'] in annotations_by_image:
                annotations_by_image[ann['image_id']].append(ann)
        print(f"✅ Loaded {len(images)} images and {len(raw_data['annotations'])} annotations.")
        return {'images': images, 'annotations': annotations_by_image, 'categories': raw_data.get('categories', [])}

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_info = self.data['images'][image_id]
        annotations = self.data['annotations'].get(image_id, [])
        
        # --- FINAL FIX: Construct path without the extra subfolder ---
        image_filename_from_json = image_info['file_name']
        # The file might be in a subfolder in the JSON, but not on disk.
        # We take only the final part of the path (the actual filename).
        base_filename = os.path.basename(image_filename_from_json)
        
        # Ensure the extension is correct (.JPEG)
        base_name_no_ext, _ = os.path.splitext(base_filename)
        correct_filename = base_name_no_ext + '.JPEG'
        
        # The full path is the specific split directory + the corrected filename.
        image_path = os.path.join(self.image_split_dir, correct_filename)
        
        global_image = self._load_global_image(image_path)
        if global_image is None: 
            return None

        part_masks = np.zeros((self.num_parts, self.img_height, self.img_width), dtype=np.float32)
        label = -1
        if annotations:
            for ann in annotations:
                part_id = ann.get('part_id')
                if part_id is not None and part_id < self.num_parts:
                    x, y, w, h = ann['bbox']
                    x_start = int((x / image_info['width']) * self.img_width)
                    y_start = int((y / image_info['height']) * self.img_height)
                    x_end = int(((x + w) / image_info['width']) * self.img_width)
                    y_end = int(((y + h) / image_info['height']) * self.img_height)
                    part_masks[part_id, y_start:y_end, x_start:x_end] = 1.0
            if annotations:
                label = int(annotations[0]['category_id'])

        combined_tensor = torch.from_numpy(np.vstack([part_masks, global_image]))
        return combined_tensor, torch.tensor(label, dtype=torch.long)

    def _load_global_image(self, image_path):
        try:
            with Image.open(image_path).convert('RGB').convert('L') as img:
                img = img.resize((self.img_width, self.img_height), Image.Resampling.LANCZOS)
            global_image = np.array(img, dtype=np.float32) / 255.0
            return np.expand_dims(global_image, axis=0)
        except FileNotFoundError:
            # This is expected for some files. The collate_fn will handle the `None` return.
            # print(f"INFO: Skipping missing image file: {image_path}")
            return None 
        except Exception as e:
            # print(f"WARNING: Could not load image {image_path}. Reason: {e}")
            return None

In [5]:
def collate_fn_skip_errors(batch):
    """A custom collate function that filters out None values from a batch."""
    # This can be useful if __getitem__ returns None on an error
    batch = list(filter(lambda x: x is not None, batch))
    if not batch:
        return torch.tensor([]), torch.tensor([])
    return torch.utils.data.dataloader.default_collate(batch)

In [6]:
class PartMaskCNN(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(PartMaskCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2), nn.Dropout(0.25),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2), nn.Dropout(0.25),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier_input_size = 128 * (IMG_HEIGHT // 8) * (IMG_WIDTH // 8)
        self.classifier = nn.Sequential(
            nn.Linear(self.classifier_input_size, 128), nn.ReLU(inplace=True),
            nn.Dropout(0.5), nn.Linear(128, num_classes),
        )
    def forward(self, x):
        x = self.features(x); x = x.view(-1, self.classifier_input_size); x = self.classifier(x); return x

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_json_path = os.path.join(ANNOTATIONS_DIR, 'train_whole', 'train.json')
with open(train_json_path, 'r') as f:
    train_data = json.load(f)
NUM_CLASSES = len(train_data['categories']) + 1
print(f"Determined {NUM_CLASSES} classes from train.json.")

print("\nLoading datasets...")
train_dataset = PartImageNetCOCODataset(
    coco_json_path=train_json_path,
    image_split_dir=os.path.join(IMAGES_DIR, 'train'), 
    num_parts=NUM_PARTS, img_dims=(IMG_HEIGHT, IMG_WIDTH)
)
val_dataset = PartImageNetCOCODataset(
    coco_json_path=os.path.join(ANNOTATIONS_DIR, 'val_whole', 'val.json'),
    image_split_dir=os.path.join(IMAGES_DIR, 'val'),
    num_parts=NUM_PARTS, img_dims=(IMG_HEIGHT, IMG_WIDTH)
)

if len(train_dataset) == 0:
    print("\n❌ FATAL: Training dataset is empty.")
else:
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, collate_fn=collate_fn_skip_errors)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, collate_fn=collate_fn_skip_errors)

    model = PartMaskCNN(in_channels=TOTAL_CHANNELS, num_classes=NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=-1)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    print("\nModel Architecture:"); print(model)

Using device: cuda
Determined 159 classes from train.json.

Loading datasets...
Loading annotations from: PartImageNet_Seg/PartImageNet/annotations/train_whole/train.json...
✅ Loaded 20481 images and 20457 annotations.
Loading annotations from: PartImageNet_Seg/PartImageNet/annotations/val_whole/val.json...
✅ Loaded 1206 images and 1205 annotations.

Model Architecture:
PartMaskCNN(
  (features): Sequential(
    (0): Conv2d(11, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Dropout(p=0.25, inplace=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Dropout(p=0.25, inplace=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): MaxPool2d(kernel_siz

In [8]:
print("\n--- Starting Training ---")

for epoch in range(EPOCHS):
    model.train(); running_loss = 0.0; train_correct = 0; train_total = 0
    for i, (inputs, labels) in enumerate(train_loader):
        if inputs.nelement() == 0: continue
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward(); optimizer.step()
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0); train_correct += (predicted == labels).sum().item()

    train_accuracy = 100 * train_correct / train_total if train_total > 0 else 0
    avg_train_loss = running_loss / len(train_loader) if len(train_loader) > 0 else 0

    model.eval(); val_correct = 0; val_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            if inputs.nelement() == 0: continue
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0); val_correct += (predicted == labels).sum().item()
    
    val_accuracy = 100 * val_correct / val_total if val_total > 0 else 0

    print(f"Epoch [{epoch+1}/{EPOCHS}] | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Train Acc: {train_accuracy:.2f}% | "
            f"Val Acc: {val_accuracy:.2f}%")

print("\n✅ Training finished successfully!")


--- Starting Training ---
Epoch [1/30] | Train Loss: 5.0685 | Train Acc: 0.58% | Val Acc: 0.83%
Epoch [2/30] | Train Loss: 5.0550 | Train Acc: 0.77% | Val Acc: 0.75%
Epoch [3/30] | Train Loss: 5.0340 | Train Acc: 0.94% | Val Acc: 1.08%
Epoch [4/30] | Train Loss: 5.0086 | Train Acc: 1.14% | Val Acc: 2.16%
Epoch [5/30] | Train Loss: 4.9739 | Train Acc: 1.58% | Val Acc: 2.24%
Epoch [6/30] | Train Loss: 4.9482 | Train Acc: 1.69% | Val Acc: 2.49%
Epoch [7/30] | Train Loss: 4.9070 | Train Acc: 1.97% | Val Acc: 2.16%
Epoch [8/30] | Train Loss: 4.8748 | Train Acc: 2.27% | Val Acc: 3.57%
Epoch [9/30] | Train Loss: 4.8454 | Train Acc: 2.18% | Val Acc: 3.40%
Epoch [10/30] | Train Loss: 4.8077 | Train Acc: 2.60% | Val Acc: 3.15%
Epoch [11/30] | Train Loss: 4.7759 | Train Acc: 2.58% | Val Acc: 3.65%


KeyboardInterrupt: 

## Independent Channel with ResNet

In [7]:
import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights

In [34]:
class PartImageNetCOCODataset(Dataset):
    def __init__(self, coco_json_path, image_split_dir, img_dims, transforms=None):
        self.coco_json_path = coco_json_path
        self.image_split_dir = image_split_dir
        self.img_height, self.img_width = img_dims
        self.transforms = transforms
        self.data, self.max_part_id, self.max_category_id = self._load_coco_data()
        self.num_parts = self.max_part_id + 1
        self.image_ids = list(self.data['images'].keys())

    def _load_coco_data(self):
        if not os.path.exists(self.coco_json_path):
            raise FileNotFoundError(f"❌ COCO JSON file not found at: {self.coco_json_path}")
        print(f"Loading annotations from: {self.coco_json_path}...")
        with open(self.coco_json_path, 'r') as f:
            raw_data = json.load(f)
        max_part_id, max_category_id = 0, 0
        for ann in raw_data['annotations']:
            max_part_id = max(max_part_id, ann.get('part_id', 0))
            max_category_id = max(max_category_id, ann.get('category_id', 0))
        images = {img['id']: img for img in raw_data['images']}
        annotations_by_image = {img_id: [] for img_id in images}
        for ann in raw_data['annotations']:
            if ann['image_id'] in annotations_by_image:
                annotations_by_image[ann['image_id']].append(ann)
        print(f"✅ Loaded {len(images)} images. Max Part ID: {max_part_id}, Max Category ID: {max_category_id}")
        return {'images': images, 'annotations': annotations_by_image}, max_part_id, max_category_id

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_info = self.data['images'][image_id]
        annotations = self.data['annotations'].get(image_id, [])
        
        image_filename = os.path.basename(image_info['file_name'])
        base_name, _ = os.path.splitext(image_filename)
        correct_filename = base_name + '.JPEG'
        image_path = os.path.join(self.image_split_dir, correct_filename)
        
        try:
            img = Image.open(image_path).convert('RGB')
            img = img.resize((self.img_width, self.img_height), Image.Resampling.LANCZOS)
        except FileNotFoundError: return None 

        part_masks = np.zeros((self.num_parts, self.img_height, self.img_width), dtype=np.float32)
        part_labels = torch.zeros(self.num_parts)
        object_label = -1
        
        if annotations:
            for ann in annotations:
                part_id = ann.get('part_id')
                if part_id is not None:
                    part_labels[part_id] = 1.0 
                    x, y, w, h = ann['bbox']
                    x_start, y_start = int((x / image_info['width']) * self.img_width), int((y / image_info['height']) * self.img_height)
                    x_end, y_end = int(((x + w) / image_info['width']) * self.img_width), int(((y + h) / image_info['height']) * self.img_height)
                    part_masks[part_id, y_start:y_end, x_start:x_end] = 1.0
            object_label = int(annotations[0]['category_id'])

        img_tensor = T.ToTensor()(img)
        if self.transforms:
            img_tensor = self.transforms(img_tensor)

        part_masks_tensor = torch.from_numpy(part_masks)
        combined_tensor = torch.cat([part_masks_tensor, img_tensor], dim=0)
        
        return combined_tensor, torch.tensor(object_label, dtype=torch.long), part_labels

In [35]:
class MultiTaskResNet(nn.Module):
    def __init__(self, in_channels, num_object_classes, num_part_classes):
        super(MultiTaskResNet, self).__init__()
        self.backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        original_conv1 = self.backbone.conv1
        original_weights = original_conv1.weight.clone()
        new_conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            new_conv1.weight[:, -3:, :, :] = original_weights 
            new_conv1.weight[:, :-3, :, :] = 0.0
        self.backbone.conv1 = new_conv1
        num_ftrs = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.object_head = nn.Linear(num_ftrs, num_object_classes)
        self.part_head = nn.Linear(num_ftrs, num_part_classes)

    def forward(self, x):
        features = self.backbone(x)
        object_output = self.object_head(features)
        part_output = self.part_head(features)
        return object_output, part_output

def collate_fn_skip_errors(batch):
    batch = list(filter(lambda x: x is not None, batch))
    if not batch: return torch.tensor([]), torch.tensor([]), torch.tensor([])
    return torch.utils.data.dataloader.default_collate(batch)

In [36]:
NUM_PARTS = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- 1. Define Data Augmentation Pipeline ---
train_transforms = T.Compose([
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    T.RandomRotation(15),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
val_transforms = T.Compose([
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# --- 2. Load Datasets and Apply Transforms ---
print("\nLoading datasets...")
train_dataset = PartImageNetCOCODataset(
    coco_json_path=os.path.join(ANNOTATIONS_DIR, 'train_whole', 'train.json'),
    image_split_dir=os.path.join(IMAGES_DIR, 'train'), 
    img_dims=(IMG_HEIGHT, IMG_WIDTH),
    transforms=train_transforms
)
val_dataset = PartImageNetCOCODataset(
    coco_json_path=os.path.join(ANNOTATIONS_DIR, 'val_whole', 'val.json'),
    image_split_dir=os.path.join(IMAGES_DIR, 'val'),
    img_dims=(IMG_HEIGHT, IMG_WIDTH),
    transforms=val_transforms
)

NUM_PARTS = train_dataset.num_parts
TOTAL_CHANNELS = NUM_PARTS + 3
NUM_CLASSES = max(train_dataset.max_category_id, val_dataset.max_category_id) + 1
print(f"Determined NUM_PARTS={NUM_PARTS} and NUM_CLASSES={NUM_CLASSES} from the data.")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, collate_fn=collate_fn_skip_errors)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, collate_fn=collate_fn_skip_errors)

model = MultiTaskResNet(
    in_channels=TOTAL_CHANNELS, 
    num_object_classes=NUM_CLASSES, 
    num_part_classes=NUM_PARTS
).to(device)

object_criterion = nn.CrossEntropyLoss(ignore_index=-1)
part_criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = StepLR(optimizer, step_size=SCHEDULER_STEP_SIZE, gamma=SCHEDULER_GAMMA)

# --- 3. Early Stopping Setup ---
best_val_accuracy = 0.0
epochs_no_improve = 0
patience = 3

print("\nModel Architecture: Adapted ResNet18"); print(model)

Using device: cuda

Loading datasets...
Loading annotations from: PartImageNet_Seg/PartImageNet/annotations/train_whole/train.json...
✅ Loaded 20481 images. Max Part ID: 0, Max Category ID: 157
Loading annotations from: PartImageNet_Seg/PartImageNet/annotations/val_whole/val.json...
✅ Loaded 1206 images. Max Part ID: 0, Max Category ID: 157
Determined NUM_PARTS=1 and NUM_CLASSES=158 from the data.

Model Architecture: Adapted ResNet18
MultiTaskResNet(
  (backbone): ResNet(
    (conv1): Conv2d(4, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_r

In [ ]:
print("\n--- Starting Training with Regularization ---")

for epoch in range(EPOCHS):
    model.train(); running_loss = 0.0; train_obj_correct = 0; train_total = 0
    for i, (inputs, object_labels, part_labels) in enumerate(train_loader):
        if inputs.nelement() == 0: continue
        inputs, object_labels, part_labels = inputs.to(device), object_labels.to(device), part_labels.to(device)
        optimizer.zero_grad()
        object_outputs, part_outputs = model(inputs)
        
        loss_object = object_criterion(object_outputs, object_labels)
        loss_part = part_criterion(part_outputs, part_labels)
        total_loss = loss_object + loss_part
        
        total_loss.backward(); optimizer.step()
        running_loss += total_loss.item()
        
        _, obj_predicted = torch.max(object_outputs.data, 1)
        train_total += object_labels.size(0)
        train_obj_correct += (obj_predicted == object_labels).sum().item()

    avg_train_loss = running_loss / len(train_loader) if len(train_loader) > 0 else 0
    train_obj_accuracy = 100 * train_obj_correct / train_total if train_total > 0 else 0

    model.eval(); val_obj_correct = 0; val_part_correct = 0; val_total = 0; total_parts = 0
    with torch.no_grad():
        for inputs, object_labels, part_labels in val_loader:
            if inputs.nelement() == 0: continue
            inputs, object_labels, part_labels = inputs.to(device), object_labels.to(device), part_labels.to(device)
            object_outputs, part_outputs = model(inputs)
            
            _, obj_predicted = torch.max(object_outputs.data, 1)
            val_total += object_labels.size(0)
            val_obj_correct += (obj_predicted == object_labels).sum().item()

            part_predicted = (torch.sigmoid(part_outputs) > 0.5).float()
            val_part_correct += (part_predicted == part_labels).sum().item()
            total_parts += part_labels.numel()

    obj_accuracy = 100 * val_obj_correct / val_total if val_total > 0 else 0
    part_accuracy = 100 * val_part_correct / total_parts if total_parts > 0 else 0

    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Train Acc: {train_obj_accuracy:.2f}% | "
            f"Val Object Acc: {obj_accuracy:.2f}% | Val Part Acc: {part_accuracy:.2f}%")

    scheduler.step()

    # --- 4. Early Stopping Check ---
    if obj_accuracy > best_val_accuracy:
        best_val_accuracy = obj_accuracy
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'best_independent_channel_model.pth')
        print(f"  -> New best validation accuracy: {best_val_accuracy:.2f}%. Model saved.")
    else:
        epochs_no_improve += 1
    
    if epochs_no_improve >= patience:
        print(f"\nEarly stopping triggered after {patience} epochs with no improvement.")
        break

print("\n✅ Training finished.")


--- Starting Training with Regularization ---


Epoch [1/30] | Train Loss: 2.3941 | Train Acc: 50.84% | Val Object Acc: 69.15% | Val Part Acc: 100.00%
  -> New best validation accuracy: 69.15%. Model saved.
Epoch [2/30] | Train Loss: 1.0915 | Train Acc: 73.11% | Val Object Acc: 71.06% | Val Part Acc: 100.00%
  -> New best validation accuracy: 71.06%. Model saved.
Epoch [3/30] | Train Loss: 0.7758 | Train Acc: 80.14% | Val Object Acc: 71.31% | Val Part Acc: 100.00%
  -> New best validation accuracy: 71.31%. Model saved.
Epoch [4/30] | Train Loss: 0.5883 | Train Acc: 84.94% | Val Object Acc: 70.32% | Val Part Acc: 100.00%
Epoch [5/30] | Train Loss: 0.4564 | Train Acc: 88.29% | Val Object Acc: 73.30% | Val Part Acc: 100.00%
  -> New best validation accuracy: 73.30%. Model saved.
Epoch [6/30] | Train Loss: 0.3561 | Train Acc: 91.28% | Val Object Acc: 71.72% | Val Part Acc: 100.00%
Epoch [7/30] | Train Loss: 0.2932 | Train Acc: 92.79% | Val Object Acc: 72.72% | Val Part Acc: 100.00%
Epoch [8/30] | Train Loss: 0.1691 | Train Acc: 96.74% |

## Independent Channel Multi Task

In [5]:
from torch.optim.lr_scheduler import StepLR

In [ ]:
class PartImageNetSharedPartsDataset(Dataset):
    """
    Correctly loads data using the provided COCO category structure.
    - Object labels are derived from 'supercategory'.
    - Shared part channels are derived by parsing the 'name' field.
    """
    def __init__(self, coco_json_path, image_split_dir, img_dims, transforms=None):
        self.coco_json_path = coco_json_path
        self.image_split_dir = image_split_dir
        self.img_height, self.img_width = img_dims
        self.transforms = transforms
        self.data, self.part_map, self.category_map, self.object_encoder = self._load_coco_data_and_build_maps()
        self.num_parts = len(self.part_map)
        self.num_classes = len(self.object_encoder)
        self.image_ids = list(self.data['images'].keys())

    def _load_coco_data_and_build_maps(self):
        if not os.path.exists(self.coco_json_path):
            raise FileNotFoundError(f"❌ COCO JSON file not found at: {self.coco_json_path}")
        
        print(f"Loading annotations from: {self.coco_json_path}...")
        with open(self.coco_json_path, 'r') as f:
            raw_data = json.load(f)

        # 1. Correctly create the Global Part-to-Channel Map
        # Extracts the last word from the part name (e.g., "Head" from "Quadruped Head")
        semantic_part_names = sorted(list(set(cat['name'].split()[-1] for cat in raw_data['categories'])))
        part_map = {name: i for i, name in enumerate(semantic_part_names)}
        
        # 2. Correctly create the Object Class Encoder from "supercategory"
        unique_supercats = sorted(list(set(cat['supercategory'] for cat in raw_data['categories'])))
        object_encoder = {name: i for i, name in enumerate(unique_supercats)}
        
        # 3. Create lookups from a part's category_id to its full info
        category_map = {cat['id']: cat for cat in raw_data['categories']}

        # 4. Create efficient data lookup tables
        images = {img['id']: img for img in raw_data['images']}
        annotations_by_image = {img_id: [] for img_id in images}
        for ann in raw_data['annotations']:
            if ann['image_id'] in annotations_by_image:
                annotations_by_image[ann['image_id']].append(ann)
        
        print(f"✅ Loaded {len(images)} images. Determined NUM_PARTS={len(part_map)} and NUM_CLASSES={len(object_encoder)}.")
        data = {'images': images, 'annotations': annotations_by_image}
        return data, part_map, category_map, object_encoder

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_info = self.data['images'][image_id]
        annotations = self.data['annotations'].get(image_id, [])
        
        image_filename_from_json = os.path.basename(image_info['file_name'])
        base_name, _ = os.path.splitext(image_filename_from_json)
        correct_filename = base_name + '.JPEG'
        image_path = os.path.join(self.image_split_dir, correct_filename)
        
        try:
            with Image.open(image_path).convert('RGB') as img:
                img_resized = img.resize((self.img_width, self.img_height), Image.Resampling.LANCZOS)
                img_np = np.array(img_resized)
        except FileNotFoundError: return None 
        except Exception as e:
            print(f"WARNING: Skipping corrupted image {image_path}. Error: {e}")
            return None

        part_masks = np.zeros((self.num_parts, self.img_height, self.img_width), dtype=np.float32)
        part_labels = torch.zeros(self.num_parts)
        object_label = -1
        
        if annotations:
            for ann in annotations:
                part_category_id = ann.get('category_id')
                part_info = self.category_map.get(part_category_id)
                
                if part_info:
                    # Use the parsed semantic name to get the correct shared channel index
                    semantic_part_name = part_info['name'].split()[-1]
                    if semantic_part_name in self.part_map:
                        channel_idx = self.part_map[semantic_part_name]
                        part_labels[channel_idx] = 1.0
                        x, y, w, h = ann['bbox']
                        x_start, y_start = int((x / image_info['width']) * self.img_width), int((y / image_info['height']) * self.img_height)
                        x_end, y_end = int(((x + w) / image_info['width']) * self.img_width), int(((y + h) / image_info['height']) * self.img_height)
                        part_masks[channel_idx, y_start:y_end, x_start:x_end] = 1.0
            
            # Use the supercategory to get the correct object label
            supercat_name = self.category_map[annotations[0]['category_id']]['supercategory']
            object_label = self.object_encoder[supercat_name]

        img_tensor = T.ToTensor()(img_np)
        if self.transforms:
            img_tensor = self.transforms(img_tensor)

        part_masks_tensor = torch.from_numpy(part_masks)
        combined_tensor = torch.cat([part_masks_tensor, img_tensor], dim=0)
        
        return combined_tensor, torch.tensor(object_label, dtype=torch.long), part_labels


In [ ]:
class MultiTaskResNet(nn.Module):
    def __init__(self, in_channels, num_object_classes, num_part_classes):
        super(MultiTaskResNet, self).__init__()
        self.backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        original_conv1 = self.backbone.conv1
        original_weights = original_conv1.weight.clone()
        new_conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            new_conv1.weight[:, -3:, :, :] = original_weights 
            new_conv1.weight[:, :-3, :, :] = 0.0
        self.backbone.conv1 = new_conv1
        num_ftrs = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.object_head = nn.Linear(num_ftrs, num_object_classes)
        self.part_head = nn.Linear(num_ftrs, num_part_classes)

    def forward(self, x):
        features = self.backbone(x)
        object_output = self.object_head(features)
        part_output = self.part_head(features)
        return object_output, part_output

def collate_fn_skip_errors(batch):
    batch = list(filter(lambda x: x is not None, batch))
    if not batch: return torch.tensor([]), torch.tensor([]), torch.tensor([])
    return torch.utils.data.dataloader.default_collate(batch)

In [ ]:
EPOCHS = 30
NUM_PARTS = 10
LEARNING_RATE = 1e-4
# --- NEW: Regularization Parameters ---
WEIGHT_DECAY = 1e-4 # L2 regularization
SCHEDULER_STEP_SIZE = 7 # Decrease LR every 7 epochs
SCHEDULER_GAMMA = 0.1 # Decrease LR by a factor of 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_transforms = T.Compose([
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    T.RandomRotation(15),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
val_transforms = T.Compose([
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("\nLoading datasets...")
train_dataset = PartImageNetCOCODataset(
    coco_json_path=os.path.join(ANNOTATIONS_DIR, 'train_whole', 'train.json'),
    image_split_dir=os.path.join(IMAGES_DIR, 'train'), 
    img_dims=(IMG_HEIGHT, IMG_WIDTH), transforms=train_transforms
)
val_dataset = PartImageNetCOCODataset(
    coco_json_path=os.path.join(ANNOTATIONS_DIR, 'val_whole', 'val.json'),
    image_split_dir=os.path.join(IMAGES_DIR, 'val'),
    img_dims=(IMG_HEIGHT, IMG_WIDTH), transforms=val_transforms
)

NUM_PARTS = train_dataset.num_parts
TOTAL_CHANNELS = NUM_PARTS + 3
NUM_CLASSES = max(train_dataset.max_category_id, val_dataset.max_category_id) + 1
print(f"Determined NUM_PARTS={NUM_PARTS} and NUM_CLASSES={NUM_CLASSES} from the data.")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, collate_fn=collate_fn_skip_errors)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, collate_fn=collate_fn_skip_errors)

model = MultiTaskResNet(
    in_channels=TOTAL_CHANNELS, 
    num_object_classes=NUM_CLASSES, 
    num_part_classes=NUM_PARTS
).to(device)

object_criterion = nn.CrossEntropyLoss(ignore_index=-1)
part_criterion = nn.BCEWithLogitsLoss()

# --- CHANGE: Add weight_decay to the optimizer ---
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# --- NEW: Initialize the learning rate scheduler ---
scheduler = StepLR(optimizer, step_size=SCHEDULER_STEP_SIZE, gamma=SCHEDULER_GAMMA)

Using device: cuda

Loading datasets...
Loading annotations from: PartImageNet_Seg/PartImageNet/annotations/train_whole/train.json...
✅ Loaded 20481 images. Max Part ID: 0, Max Category ID: 157
Loading annotations from: PartImageNet_Seg/PartImageNet/annotations/val_whole/val.json...
✅ Loaded 1206 images. Max Part ID: 0, Max Category ID: 157
Determined NUM_PARTS=1 and NUM_CLASSES=158 from the data.


In [ ]:
print("\n--- Starting Multi-Task Training with Regularization ---")

for epoch in range(EPOCHS):
    model.train(); running_loss = 0.0; train_obj_correct = 0; train_total = 0
    for i, (inputs, object_labels, part_labels) in enumerate(train_loader):
        if inputs.nelement() == 0: continue
        inputs, object_labels, part_labels = inputs.to(device), object_labels.to(device), part_labels.to(device)
        optimizer.zero_grad()
        object_outputs, part_outputs = model(inputs)
        
        loss_object = object_criterion(object_outputs, object_labels)
        loss_part = part_criterion(part_outputs, part_labels)
        total_loss = loss_object + loss_part
        
        total_loss.backward()
        optimizer.step()
        running_loss += total_loss.item()
        
        _, obj_predicted = torch.max(object_outputs.data, 1)
        train_total += object_labels.size(0)
        train_obj_correct += (obj_predicted == object_labels).sum().item()

    avg_train_loss = running_loss / len(train_loader) if len(train_loader) > 0 else 0
    train_obj_accuracy = 100 * train_obj_correct / train_total if train_total > 0 else 0

    model.eval(); val_obj_correct = 0; val_part_correct = 0; val_total = 0; total_parts = 0
    with torch.no_grad():
        for inputs, object_labels, part_labels in val_loader:
            if inputs.nelement() == 0: continue
            inputs, object_labels, part_labels = inputs.to(device), object_labels.to(device), part_labels.to(device)
            object_outputs, part_outputs = model(inputs)
            
            _, obj_predicted = torch.max(object_outputs.data, 1)
            val_total += object_labels.size(0)
            val_obj_correct += (obj_predicted == object_labels).sum().item()

            part_predicted = (torch.sigmoid(part_outputs) > 0.5).float()
            val_part_correct += (part_predicted == part_labels).sum().item()
            total_parts += part_labels.numel()

    obj_accuracy = 100 * val_obj_correct / val_total if val_total > 0 else 0
    part_accuracy = 100 * val_part_correct / total_parts if total_parts > 0 else 0

    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Train Acc: {train_obj_accuracy:.2f}% | "
            f"Val Object Acc: {obj_accuracy:.2f}% | "
            f"Val Part Acc: {part_accuracy:.2f}%")

    # --- NEW: Step the learning rate scheduler ---
    scheduler.step()

print("\n✅ Training finished.")


--- Starting Multi-Task Training with Regularization ---
Epoch [1/30] | Train Loss: 2.3847 | Train Acc: 50.91% | Val Object Acc: 68.08% | Val Part Acc: 100.00%
Epoch [2/30] | Train Loss: 1.0860 | Train Acc: 73.45% | Val Object Acc: 71.56% | Val Part Acc: 100.00%
Epoch [3/30] | Train Loss: 0.7728 | Train Acc: 80.43% | Val Object Acc: 72.55% | Val Part Acc: 100.00%
Epoch [4/30] | Train Loss: 0.5932 | Train Acc: 84.63% | Val Object Acc: 73.80% | Val Part Acc: 100.00%
Epoch [5/30] | Train Loss: 0.4538 | Train Acc: 88.37% | Val Object Acc: 72.64% | Val Part Acc: 100.00%
Epoch [6/30] | Train Loss: 0.3623 | Train Acc: 90.87% | Val Object Acc: 72.80% | Val Part Acc: 100.00%
Epoch [7/30] | Train Loss: 0.2843 | Train Acc: 93.23% | Val Object Acc: 71.64% | Val Part Acc: 100.00%
Epoch [8/30] | Train Loss: 0.1668 | Train Acc: 96.96% | Val Object Acc: 74.79% | Val Part Acc: 100.00%
Epoch [9/30] | Train Loss: 0.1297 | Train Acc: 97.91% | Val Object Acc: 75.87% | Val Part Acc: 100.00%
Epoch [10/30] |

## Shared Channel with ResNet

In [3]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
SCHEDULER_STEP_SIZE = 7
SCHEDULER_GAMMA = 0.1

import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights
from torch.optim.lr_scheduler import StepLR

In [21]:
class PartImageNetSharedPartsDataset(Dataset):
    def __init__(self, coco_json_path, image_split_dir, img_dims, transforms=None):
        self.coco_json_path = coco_json_path
        self.image_split_dir = image_split_dir
        self.img_height, self.img_width = img_dims
        self.transforms = transforms
        self.data, self.part_map, self.category_map, self.object_encoder = self._load_coco_data_and_build_maps()
        self.num_parts = len(self.part_map)
        self.num_classes = len(self.object_encoder)
        self.image_ids = list(self.data['images'].keys())

    def _load_coco_data_and_build_maps(self):
        if not os.path.exists(self.coco_json_path):
            raise FileNotFoundError(f"❌ COCO JSON file not found at: {self.coco_json_path}")
        
        print(f"Loading annotations from: {self.coco_json_path}...")
        with open(self.coco_json_path, 'r') as f:
            raw_data = json.load(f)

        # --- DEBUGGING SECTION ---
        print("\n--- Running Part Name Parsing Debugger ---")
        semantic_part_names = set()
        for i, cat in enumerate(raw_data['categories']):
            supercategory = cat['supercategory']
            full_part_name = cat['name']
            
            # This is the new, more robust parsing logic.
            # It only considers it a "part" if the supercategory is in the name.
            if supercategory in full_part_name:
                semantic_part = full_part_name.replace(supercategory, '').strip()
                semantic_part_names.add(semantic_part)
                if i < 10: # Print the first 10 successful examples
                    print(f"  - SUCCESS: Parsed '{full_part_name}' -> '{semantic_part}'")
            else:
                if i < 10: # Print the first 10 ignored examples
                    print(f"  - INFO: Ignoring '{full_part_name}' as it's likely an object category, not a part.")

        part_map = {name: i for i, name in enumerate(sorted(list(semantic_part_names)))}
        print("--- Final Mapping ---")
        print(f"  - Total unique semantic parts found: {len(part_map)}")
        print(f"  - Part Map: {part_map}")
        print("--------------------------\n")
        # --- END DEBUGGING SECTION ---

        unique_supercats = sorted(list(set(cat['supercategory'] for cat in raw_data['categories'])))
        object_encoder = {name: i for i, name in enumerate(unique_supercats)}
        
        category_map = {cat['id']: cat for cat in raw_data['categories']}

        images = {img['id']: img for img in raw_data['images']}
        annotations_by_image = {img_id: [] for img_id in images}
        for ann in raw_data['annotations']:
            if ann['image_id'] in annotations_by_image:
                annotations_by_image[ann['image_id']].append(ann)
        
        print(f"✅ Loaded {len(images)} images. Determined NUM_PARTS={len(part_map)} and NUM_CLASSES={len(object_encoder)}.")
        data = {'images': images, 'annotations': annotations_by_image}
        return data, part_map, category_map, object_encoder

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_info = self.data['images'][image_id]
        annotations = self.data['annotations'].get(image_id, [])
        
        image_filename_from_json = os.path.basename(image_info['file_name'])
        base_name, _ = os.path.splitext(image_filename_from_json)
        correct_filename = base_name + '.JPEG'
        image_path = os.path.join(self.image_split_dir, correct_filename)
        
        try:
            with Image.open(image_path).convert('RGB') as img:
                img_resized = img.resize((self.img_width, self.img_height), Image.Resampling.LANCZOS)
                img_np = np.array(img_resized)
        except FileNotFoundError: return None 
        except Exception as e:
            print(f"WARNING: Skipping corrupted image {image_path}. Error: {e}")
            return None

        part_masks = np.zeros((self.num_parts, self.img_height, self.img_width), dtype=np.float32)
        part_labels = torch.zeros(self.num_parts)
        object_label = -1
        
        if annotations:
            for ann in annotations:
                part_category_id = ann.get('category_id')
                part_info = self.category_map.get(part_category_id)
                
                if part_info and part_info['supercategory'] in part_info['name']:
                    semantic_part_name = part_info['name'].replace(part_info['supercategory'], '').strip()
                    if semantic_part_name in self.part_map:
                        channel_idx = self.part_map[semantic_part_name]
                        part_labels[channel_idx] = 1.0
                        x, y, w, h = ann['bbox']
                        x_start, y_start = int((x / image_info['width']) * self.img_width), int((y / image_info['height']) * self.img_height)
                        x_end, y_end = int(((x + w) / image_info['width']) * self.img_width), int(((y + h) / image_info['height']) * self.img_height)
                        part_masks[channel_idx, y_start:y_end, x_start:x_end] = 1.0
            
            supercat_name = self.category_map[annotations[0]['category_id']]['supercategory']
            object_label = self.object_encoder[supercat_name]

        img_tensor = T.ToTensor()(img_np)
        if self.transforms:
            img_tensor = self.transforms(img_tensor)

        part_masks_tensor = torch.from_numpy(part_masks)
        combined_tensor = torch.cat([part_masks_tensor, img_tensor], dim=0)
        
        return combined_tensor, torch.tensor(object_label, dtype=torch.long), part_labels


In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_transforms = T.Compose([
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
val_transforms = T.Compose([
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("\nLoading datasets...")
train_dataset = PartImageNetSharedPartsDataset(
    coco_json_path=os.path.join(ANNOTATIONS_DIR, 'train_whole', 'train.json'),
    image_split_dir=os.path.join(IMAGES_DIR, 'train'), 
    img_dims=(IMG_HEIGHT, IMG_WIDTH), transforms=train_transforms
)
val_dataset = PartImageNetSharedPartsDataset(
    coco_json_path=os.path.join(ANNOTATIONS_DIR, 'val_whole', 'val.json'),
    image_split_dir=os.path.join(IMAGES_DIR, 'val'),
    img_dims=(IMG_HEIGHT, IMG_WIDTH), transforms=val_transforms
)

NUM_PARTS = train_dataset.num_parts
TOTAL_CHANNELS = NUM_PARTS + 3
NUM_CLASSES = train_dataset.num_classes
print(f"Using dynamically determined NUM_PARTS={NUM_PARTS} and NUM_CLASSES={NUM_CLASSES}.")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate_fn_skip_errors)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_fn_skip_errors)

model = MultiTaskResNet(
    in_channels=TOTAL_CHANNELS, 
    num_object_classes=NUM_CLASSES, 
    num_part_classes=NUM_PARTS
).to(device)

object_criterion = nn.CrossEntropyLoss(ignore_index=-1)
part_criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = StepLR(optimizer, step_size=SCHEDULER_STEP_SIZE, gamma=SCHEDULER_GAMMA)

best_val_accuracy = 0.0
epochs_no_improve = 0
patience = 5

Using device: cuda

Loading datasets...
Loading annotations from: PartImageNet_Seg/PartImageNet/annotations/train_whole/train.json...

--- Running Part Name Parsing Debugger ---
  - INFO: Ignoring 'n02422699' as it's likely an object category, not a part.
  - INFO: Ignoring 'n01740131' as it's likely an object category, not a part.
  - INFO: Ignoring 'n01692333' as it's likely an object category, not a part.
  - INFO: Ignoring 'n01734418' as it's likely an object category, not a part.
  - INFO: Ignoring 'n04483307' as it's likely an object category, not a part.
  - INFO: Ignoring 'n02514041' as it's likely an object category, not a part.
  - INFO: Ignoring 'n02058221' as it's likely an object category, not a part.
  - INFO: Ignoring 'n03417042' as it's likely an object category, not a part.
  - INFO: Ignoring 'n02814533' as it's likely an object category, not a part.
  - INFO: Ignoring 'n02124075' as it's likely an object category, not a part.
--- Final Mapping ---
  - Total unique sem

/home/xjzb2/miniconda3/envs/hycoclip/lib/python3.9/site-packages/torch/nn/init.py:511: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


In [10]:
class PartImageNetSharedPartsDataset(Dataset):
    def __init__(self, coco_json_path, image_split_dir, img_dims, transforms=None):
        self.coco_json_path = coco_json_path
        self.image_split_dir = image_split_dir
        self.img_height, self.img_width = img_dims
        self.transforms = transforms
        self.data, self.part_map, self.category_map, self.object_encoder = self._load_coco_data_and_build_maps()
        self.num_parts = len(self.part_map)
        self.num_classes = len(self.object_encoder)
        self.image_ids = list(self.data['images'].keys())

    def _load_coco_data_and_build_maps(self):
        if not os.path.exists(self.coco_json_path):
            raise FileNotFoundError(f"❌ COCO JSON file not found at: {self.coco_json_path}")
        
        print(f"Loading annotations from: {self.coco_json_path}...")
        with open(self.coco_json_path, 'r') as f:
            raw_data = json.load(f)

        # --- FINAL FIX: Robustly parse semantic part names ---
        semantic_part_names = set()
        for cat in raw_data['categories']:
            # Replace the supercategory name from the part name and strip whitespace
            # e.g., "Car Side Mirror".replace("Car", "") -> " Side Mirror" -> "Side Mirror"
            part_name = cat['name'].replace(cat['supercategory'], '').strip()
            semantic_part_names.add(part_name)
        
        part_map = {name: i for i, name in enumerate(sorted(list(semantic_part_names)))}
        
        unique_supercats = sorted(list(set(cat['supercategory'] for cat in raw_data['categories'])))
        object_encoder = {name: i for i, name in enumerate(unique_supercats)}
        
        category_map = {cat['id']: cat for cat in raw_data['categories']}

        images = {img['id']: img for img in raw_data['images']}
        annotations_by_image = {img_id: [] for img_id in images}
        for ann in raw_data['annotations']:
            if ann['image_id'] in annotations_by_image:
                annotations_by_image[ann['image_id']].append(ann)
        
        print(f"✅ Loaded {len(images)} images. Determined NUM_PARTS={len(part_map)} and NUM_CLASSES={len(object_encoder)}.")
        data = {'images': images, 'annotations': annotations_by_image}
        return data, part_map, category_map, object_encoder

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_info = self.data['images'][image_id]
        annotations = self.data['annotations'].get(image_id, [])
        
        image_filename_from_json = os.path.basename(image_info['file_name'])
        base_name, _ = os.path.splitext(image_filename_from_json)
        correct_filename = base_name + '.JPEG'
        image_path = os.path.join(self.image_split_dir, correct_filename)
        
        try:
            with Image.open(image_path).convert('RGB') as img:
                img_resized = img.resize((self.img_width, self.img_height), Image.Resampling.LANCZOS)
                img_np = np.array(img_resized)
        except FileNotFoundError: return None 
        except Exception as e:
            print(f"WARNING: Skipping corrupted image {image_path}. Error: {e}")
            return None

        part_masks = np.zeros((self.num_parts, self.img_height, self.img_width), dtype=np.float32)
        part_labels = torch.zeros(self.num_parts)
        object_label = -1
        
        if annotations:
            for ann in annotations:
                part_category_id = ann.get('category_id')
                part_info = self.category_map.get(part_category_id)
                
                if part_info:
                    # Use the same robust parsing logic to find the channel index
                    semantic_part_name = part_info['name'].replace(part_info['supercategory'], '').strip()
                    if semantic_part_name in self.part_map:
                        channel_idx = self.part_map[semantic_part_name]
                        part_labels[channel_idx] = 1.0
                        x, y, w, h = ann['bbox']
                        x_start, y_start = int((x / image_info['width']) * self.img_width), int((y / image_info['height']) * self.img_height)
                        x_end, y_end = int(((x + w) / image_info['width']) * self.img_width), int(((y + h) / image_info['height']) * self.img_height)
                        part_masks[channel_idx, y_start:y_end, x_start:x_end] = 1.0
            
            supercat_name = self.category_map[annotations[0]['category_id']]['supercategory']
            object_label = self.object_encoder[supercat_name]

        img_tensor = T.ToTensor()(img_np)
        if self.transforms:
            img_tensor = self.transforms(img_tensor)

        part_masks_tensor = torch.from_numpy(part_masks)
        combined_tensor = torch.cat([part_masks_tensor, img_tensor], dim=0)
        
        return combined_tensor, torch.tensor(object_label, dtype=torch.long), part_labels


In [5]:
class MultiTaskResNet(nn.Module):
    def __init__(self, in_channels, num_object_classes, num_part_classes):
        super(MultiTaskResNet, self).__init__()
        self.backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        original_conv1 = self.backbone.conv1
        original_weights = original_conv1.weight.clone()
        new_conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            new_conv1.weight[:, -3:, :, :] = original_weights 
            new_conv1.weight[:, :-3, :, :] = 0.0
        self.backbone.conv1 = new_conv1
        num_ftrs = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.object_head = nn.Linear(num_ftrs, num_object_classes)
        self.part_head = nn.Linear(num_ftrs, num_part_classes)

    def forward(self, x):
        features = self.backbone(x)
        object_output = self.object_head(features)
        part_output = self.part_head(features)
        return object_output, part_output

def collate_fn_skip_errors(batch):
    batch = list(filter(lambda x: x is not None, batch))
    if not batch: return torch.tensor([]), torch.tensor([]), torch.tensor([])
    return torch.utils.data.dataloader.default_collate(batch)

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_transforms = T.Compose([
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
val_transforms = T.Compose([
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("\nLoading datasets...")
train_dataset = PartImageNetSharedPartsDataset(
    coco_json_path=os.path.join(ANNOTATIONS_DIR, 'train_whole', 'train.json'),
    image_split_dir=os.path.join(IMAGES_DIR, 'train'), 
    img_dims=(IMG_HEIGHT, IMG_WIDTH), transforms=train_transforms
)
val_dataset = PartImageNetSharedPartsDataset(
    coco_json_path=os.path.join(ANNOTATIONS_DIR, 'val_whole', 'val.json'),
    image_split_dir=os.path.join(IMAGES_DIR, 'val'),
    img_dims=(IMG_HEIGHT, IMG_WIDTH), transforms=val_transforms
)

NUM_PARTS = train_dataset.num_parts
TOTAL_CHANNELS = NUM_PARTS + 3
NUM_CLASSES = train_dataset.num_classes
print(f"Using dynamically determined NUM_PARTS={NUM_PARTS} and NUM_CLASSES={NUM_CLASSES}.")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate_fn_skip_errors)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_fn_skip_errors)

model = MultiTaskResNet(
    in_channels=TOTAL_CHANNELS, 
    num_object_classes=NUM_CLASSES, 
    num_part_classes=NUM_PARTS
).to(device)

object_criterion = nn.CrossEntropyLoss(ignore_index=-1)
part_criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = StepLR(optimizer, step_size=SCHEDULER_STEP_SIZE, gamma=SCHEDULER_GAMMA)

best_val_accuracy = 0.0
epochs_no_improve = 0
patience = 5

Using device: cuda

Loading datasets...
Loading annotations from: PartImageNet_Seg/PartImageNet/annotations/train_whole/train.json...
✅ Loaded 20481 images. Determined NUM_PARTS=158 and NUM_CLASSES=11.
Loading annotations from: PartImageNet_Seg/PartImageNet/annotations/val_whole/val.json...
✅ Loaded 1206 images. Determined NUM_PARTS=158 and NUM_CLASSES=11.
Using dynamically determined NUM_PARTS=158 and NUM_CLASSES=11.


In [15]:
print("\n--- Starting Multi-Task Training with Regularization ---")

for epoch in range(EPOCHS):
    model.train(); running_loss = 0.0; train_obj_correct = 0; train_total = 0
    
    # --- FIX: Unpack all three items from the DataLoader ---
    for i, (inputs, object_labels, part_labels) in enumerate(train_loader):
        if inputs.nelement() == 0: continue
        inputs, object_labels, part_labels = inputs.to(device), object_labels.to(device), part_labels.to(device)
        optimizer.zero_grad()
        object_outputs, part_outputs = model(inputs)
        
        loss_object = object_criterion(object_outputs, object_labels)
        loss_part = part_criterion(part_outputs, part_labels)
        total_loss = loss_object + loss_part
        
        total_loss.backward(); optimizer.step()
        running_loss += total_loss.item()
        
        _, obj_predicted = torch.max(object_outputs.data, 1)
        train_total += object_labels.size(0)
        train_obj_correct += (obj_predicted == object_labels).sum().item()

    avg_train_loss = running_loss / len(train_loader) if len(train_loader) > 0 else 0
    train_obj_accuracy = 100 * train_obj_correct / train_total if train_total > 0 else 0

    model.eval(); val_obj_correct = 0; val_part_correct = 0; val_total = 0; total_parts = 0
    with torch.no_grad():
        for inputs, object_labels, part_labels in val_loader:
            if inputs.nelement() == 0: continue
            inputs, object_labels, part_labels = inputs.to(device), object_labels.to(device), part_labels.to(device)
            object_outputs, part_outputs = model(inputs)
            
            _, obj_predicted = torch.max(object_outputs.data, 1)
            val_total += object_labels.size(0)
            val_obj_correct += (obj_predicted == object_labels).sum().item()

            part_predicted = (torch.sigmoid(part_outputs) > 0.5).float()
            val_part_correct += (part_predicted == part_labels).sum().item()
            total_parts += part_labels.numel()

    obj_accuracy = 100 * val_obj_correct / val_total if val_total > 0 else 0
    part_accuracy = 100 * val_part_correct / total_parts if total_parts > 0 else 0

    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Train Acc: {train_obj_accuracy:.2f}% | "
            f"Val Object Acc: {obj_accuracy:.2f}% | Val Part Acc: {part_accuracy:.2f}%")

    scheduler.step()

    if obj_accuracy > best_val_accuracy:
        best_val_accuracy = obj_accuracy
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'best_independent_channel_model.pth')
        print(f"  -> New best validation accuracy: {best_val_accuracy:.2f}%. Model saved.")
    else:
        epochs_no_improve += 1
    
    if epochs_no_improve >= patience:
        print(f"\nEarly stopping triggered after {patience} epochs with no improvement.")
        break

print("\n✅ Training finished.")


--- Starting Multi-Task Training with Regularization ---


ERROR: Unexpected bus error encountered in worker. This might be caused by insufficient shared memory (shm).
 ERROR: Unexpected bus error encountered in worker. This might be caused by insufficient shared memory (shm).
 

RuntimeError: DataLoader worker (pid 2874) is killed by signal: Bus error. It is possible that dataloader's workers are out of shared memory. Please try to raise your shared memory limit.

In [ ]:
save_path = './shared_resnet.pth'
        
# Save the model's state_dict
torch.save(model.state_dict(), save_path)

print(f"✅ Model saved to {save_path}")

✅ Model saved to ./shared_resnet.pth


## Cue-Conflict

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from modelvshuman.models.wrappers.pytorch import PytorchModel
from modelvshuman.evaluation import evaluate_model

2025-07-25 22:44:21.132806: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753454661.366938     973 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753454661.425651     973 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1753454661.922641     973 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1753454661.922722     973 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1753454661.922725     973 computation_placer.cc:177] computation placer alr

ImportError: cannot import name 'evaluate_model' from 'modelvshuman.evaluation' (/home/xjzb2/compo_learning/model-vs-human/modelvshuman/evaluation/__init__.py)